# NYC 311 HPD Complaint Resolution — Data Cleaning & Prep

Phase 1 of the ticket-resolution process-improvement project. Loads the raw NYC 311 HPD complaint sample, applies the "valid closed record" rule defined in the BRD, flags exact-duplicate records for the sensitivity analysis (BR/Constraints), and adds the 30-day analytical-threshold breach flag used in BR-002.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/nyc311_hpd_complaints_v2.csv")
df.shape

(405, 7)

## Parse dates and define valid closed records

Per the BRD Constraints section: a *valid closed record* has `status = Closed`, a non-null `closed_date`, and a non-negative `resolution_time_hours`.

In [2]:
df["created_date"] = pd.to_datetime(df["created_date"])
df["closed_date"] = pd.to_datetime(df["closed_date"])

df["is_valid_closed"] = (
    (df["status"] == "Closed")
    & df["closed_date"].notna()
    & (df["resolution_time_hours"] >= 0)
)

print("Valid closed records:", df["is_valid_closed"].sum())
print("Excluded records:", (~df["is_valid_closed"]).sum())

Valid closed records: 405
Excluded records: 0


## Flag exact-duplicate records

Records sharing identical `created_date`, `closed_date`, and `complaint_type` are flagged. The first record in each group is kept as the representative; the rest are marked as excess rows for the BRD's sensitivity analysis.

In [3]:
dup_key = ["created_date", "closed_date", "complaint_type"]
df["exact_dup_group_size"] = df.groupby(dup_key)["unique_key"].transform("count")
df["is_exact_duplicate"] = df["exact_dup_group_size"] > 1

df["_rank_in_group"] = df.groupby(dup_key).cumcount()
df["is_excess_duplicate_row"] = df["is_exact_duplicate"] & (df["_rank_in_group"] > 0)
df.drop(columns=["_rank_in_group"], inplace=True)

n_groups = df.loc[df["is_exact_duplicate"], dup_key].drop_duplicates().shape[0]
print("Exact-duplicate groups:", n_groups)
print("Records involved:", df["is_exact_duplicate"].sum())
print("Excess rows (sensitivity analysis candidates):", df["is_excess_duplicate_row"].sum())

Exact-duplicate groups: 31
Records involved: 68
Excess rows (sensitivity analysis candidates): 37


## BR-002: 30-day analytical-threshold flag

In [4]:
df["breaches_30day_threshold"] = df["resolution_time_days"] > 30
df["breaches_30day_threshold"].value_counts()

breaches_30day_threshold
False    336
True      69
Name: count, dtype: int64

## Quick sanity check: resolution time by complaint type (BR-001)

Mean, median, and 90th percentile — computed here in pandas as a preview; the authoritative version will be built in SQL in Phase 3.

In [5]:
summary = df.groupby("complaint_type")["resolution_time_days"].agg(
    ticket_count="count",
    mean_days="mean",
    median_days="median",
    p90_days=lambda x: x.quantile(0.90)
).round(2).sort_values("median_days", ascending=False)

summary

,ticket_count,mean_days,median_days,p90_days
complaint_type,,,,
APPLIANCE,15,29.39,18.10,64.87
PAINT/PLASTER,43,28.95,16.26,82.06
ELEVATOR,1,15.85,15.85,15.85
DOOR/WINDOW,31,19.72,15.38,31.08
ELECTRIC,25,18.99,14.35,52.19
WATER LEAK,32,22.01,10.31,70.54
GENERAL,22,34.14,9.80,116.76
FLOORING/STAIRS,28,22.93,9.45,82.86
UNSANITARY CONDITION,87,15.77,9.03,36.84


## Save cleaned dataset

In [6]:
df["created_day"] = df["created_date"].dt.date
df["created_dayofweek"] = df["created_date"].dt.day_name()

df.to_csv("../data/nyc311_hpd_complaints_cleaned.csv", index=False)
print("Saved. Shape:", df.shape)

Saved. Shape: (405, 14)
